# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

1) Unit of analysis: one row = one content item × one reporting day (report_date × client_hash_id × content_hash_id).

2) Table(s): `fact_content_daily_performance` (daily fact) joined to `dim_content` for static content metadata.

3) Time window for this contract: a mid-panel month — `month = 2026-03` (decision moment = 2026-03-31).

4) Label / proxy: `is_declining_label` — proxy defined here as impressions declining in the last 30 days vs the previous 30 days (derived, used as a simple proxy for 'decline').

5) Excluded: product decision flags or any `*_last30` fields from `fact_content_query_90d` that overlap the label period — excluded because they can contain label-period information (leakage).

In [1]:
# Connect to the Hugging Face release via DuckDB (uses HF_TOKEN from env or prompt).
import os, getpass
import duckdb
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'
# Use the month partition for 2026-03 to avoid scanning the whole table repeatedly
FACT_MONTH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
print('DuckDB secret created — ready to run queries on month=2026-03')

DuckDB secret created — ready to run queries on month=2026-03


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [2]:
# 3a) Grain check: confirm one row per (report_date, client_hash_id, content_hash_id)
q_grain = f"SELECT client_hash_id, content_hash_id, COUNT(*) AS c FROM {FACT_MONTH} GROUP BY 1,2 HAVING c > 1 LIMIT 5"
print('Running grain check...')
print(con.sql(q_grain).df())

# 3b) Row count and date span for the selected month partition
q_counts = f"SELECT COUNT(*) AS rows, MIN(report_date) AS min_d, MAX(report_date) AS max_d FROM {FACT_MONTH}"
print('\nMonth counts and window:')
print(con.sql(q_counts).df())

# 3c) Availability check: show rows where GA4 data is available (filter with IS TRUE)
q_avail = f"SELECT COUNT(*) AS rows_with_ga4 FROM {FACT_MONTH} WHERE ga4_data_available IS TRUE"
print('\nGA4 availability (IS TRUE):')
print(con.sql(q_avail).df())


Running grain check...


            client_hash_id           content_hash_id   c
0  client_e547b89c05043229  content_c77e7d7e021340e8  31
1  client_e547b89c05043229  content_545bb6cc7081ded3  31
2  client_e547b89c05043229  content_e81831e1e8f621f2  31
3  client_e547b89c05043229  content_6c3009b2bbb91c33  31
4  client_e547b89c05043229  content_9d82eac28dba686f  31

Month counts and window:


      rows      min_d      max_d
0  9841378 2026-03-01 2026-03-31

GA4 availability (IS TRUE):


   rows_with_ga4
0         413966


# 4) Build a small 5-feature frame for month=2026-03 (features are knowable before decision moment)
# Decision moment: 2026-03-31. Feature window: previous 30 days before decision (prev30).
frame_q = f"













del leak_df['leaky_label']print('Dropping the leaky column for subsequent honest modeling.')print(leak_df.head())print('Leak sample:')leak_df['leaky_label'] = (leak_df['imp_prev30'] < leak_df['imp_prev30'].median()).astype(int)leak_df = features_df.copy()# Leakage demo: create one label-derived column on purpose, inspect impact, then remove itfeatures_df.head()print(features_df.shape)features_df = con.sql(frame_q).df()print('Building feature frame (first 200 rows)...')WITH bounds AS (SELECT DATE '2026-03-31' AS end_d),
windowed AS (
  SELECT f.client_hash_id, f.content_hash_id,
    SUM(CASE WHEN f.report_date > b.end_d - INTERVAL 60 DAY AND f.report_date <= b.end_d - INTERVAL 31 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
    SUM(CASE WHEN f.report_date > b.end_d - INTERVAL 60 DAY AND f.report_date <= b.end_d - INTERVAL 31 DAY THEN f.gsc_clicks ELSE 0 END) AS clk_prev30,
    AVG(CASE WHEN f.report_date > b.end_d - INTERVAL 60 DAY AND f.report_date <= b.end_d - INTERVAL 31 DAY THEN f.gsc_avg_position END) AS pos_prev30,
    ANY_VALUE(d.word_count) AS word_count,
    SUM(CASE WHEN f.report_date > b.end_d - INTERVAL 60 DAY AND f.report_date <= b.end_d - INTERVAL 31 DAY THEN f.sessions_ai ELSE 0 END) AS ai_sessions_prev30
  FROM {FACT_MONTH} f, bounds b
  LEFT JOIN {DIM_CONTENT} d ON f.content_hash_id = d.content_hash_id
  GROUP BY 1,2
  HAVING imp_prev30 >= 50
)
SELECT * FROM windowed LIMIT 200
"

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.